Air Traffic MDP Simulation
===========================
Mesa-based simulation of aircraft navigating around TFR zones during space launches.

Project: MDP/POMDP Air Traffic Navigation During Space Launch Operations
Phase:   2 — Baseline Policies + Scaling

What this file does:
- Loads ALL .csv files from a folder automatically (handles both single and combined files)
- Fills data gaps > 60 seconds with interpolation (ADS-B dead zones)
- Runs simulation in TWO modes you can compare:
    'replay' = aircraft follow recorded CSV data exactly (historical baseline)
    'policy' = aircraft use rule-based decisions to avoid TFR and conflicts
- Fixes the repeated violation problem (only logs when a violation STARTS)
- Predicts conflicts 20 steps ahead so aircraft can react early
- Plotly map uses ACTUAL simulation history, not just raw CSV data

In [1]:
import pandas as pd
import numpy as np
import os
import webbrowser
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from mesa import Agent, Model
from mesa.time import BaseScheduler

In [1]:
# CONFIGURATION — change these to match your setup

DATA_FOLDER = r"C:\Users\Dev Sharma\Desktop\EREU\Datasets\Validation data (FlightRader24)"

# TFR zone: Starship launch site (Cape Canaveral / Kennedy Space Center)
TFR_LAT       = 28.4889
TFR_LON       = -80.5778
TFR_RADIUS_NM = 30

# If time between two ADS-B rows is MORE than this, we interpolate the gap
GAP_THRESHOLD_SECONDS = 60

# How many simulation steps ahead agents look for conflicts
CONFLICT_LOOKAHEAD_STEPS = 20

# Time each simulation step represents (seconds of real flight)
STEP_DURATION_SECONDS = 30

In [3]:
# GEOMETRY HELPERS

def haversine(lat1, lon1, lat2, lon2):
    """
    Distance between two lat/lon points in NAUTICAL MILES.
    Uses the Haversine formula — handles the curvature of the Earth.
    """
    R = 3440  # Earth radius in nm

    # Convert degrees to radians (math functions need radians)
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def bearing_between(lat1, lon1, lat2, lon2):
    """
    Compass bearing FROM point 1 TO point 2 (degrees, 0 = North, 90 = East).
    This tells an aircraft which direction to fly to get from A to B.
    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)

    bearing = np.degrees(np.arctan2(x, y))
    return (bearing + 360) % 360  # normalise to 0–360


def move_by_heading(lat, lon, heading_deg, speed_kts, time_seconds):
    """
    Dead reckoning: given a position, heading and speed, compute new position
    after flying for `time_seconds`.

    heading_deg : compass direction (0 = North, 90 = East)
    speed_kts   : speed in knots (= nautical miles per hour)
    time_seconds: how long we fly

    Returns: (new_lat, new_lon)
    """
    R = 3440  # nm

    speed_nm_per_sec = speed_kts / 3600.0
    distance_nm      = speed_nm_per_sec * time_seconds

    lat_rad     = np.radians(lat)
    lon_rad     = np.radians(lon)
    heading_rad = np.radians(heading_deg)

    angular_dist = distance_nm / R  # fraction of Earth's radius

    new_lat_rad = np.arcsin(
        np.sin(lat_rad) * np.cos(angular_dist)
        + np.cos(lat_rad) * np.sin(angular_dist) * np.cos(heading_rad)
    )

    new_lon_rad = lon_rad + np.arctan2(
        np.sin(heading_rad) * np.sin(angular_dist) * np.cos(lat_rad),
        np.cos(angular_dist) - np.sin(lat_rad) * np.sin(new_lat_rad),
    )

    return np.degrees(new_lat_rad), np.degrees(new_lon_rad)

In [4]:
# GAP FILLING (ADS-B DEAD ZONES)

def fill_gaps(df, gap_threshold_seconds=60):
    """
    ADS-B receivers have blind spots — aircraft disappear for stretches > 60 sec.
    Rather than having the simulation agent "jump" between positions,
    we linearly interpolate through the gap at 30-second intervals.

    Each interpolated row is flagged with interpolated=True so we can
    track how much of each flight was estimated vs observed.

    This is the simple baseline for dead zones.
    Later: POMDP uncertainty modelling can replace the interpolation here,
    treating the gap as a region of uncertain belief state.
    """
    rows = []

    for i in range(len(df) - 1):
        rows.append(df.iloc[i].to_dict())

        current_time = df["UTC"].iloc[i]
        next_time    = df["UTC"].iloc[i + 1]
        gap_sec      = (next_time - current_time).total_seconds()

        if gap_sec > gap_threshold_seconds:
            # How many 30-second fill steps fit inside this gap?
            num_fill = int(gap_sec / STEP_DURATION_SECONDS) - 1
            print(f"    Gap of {gap_sec:.0f}s detected — inserting {num_fill} interpolated points")

            for step in range(1, num_fill + 1):
                # fraction: 0.0 = at current row, 1.0 = at next row
                frac = (step * STEP_DURATION_SECONDS) / gap_sec

                fill = df.iloc[i].to_dict()  # start from current row's values
                fill["UTC"]          = current_time + pd.Timedelta(seconds=step * STEP_DURATION_SECONDS)
                fill["lat"]          = df["lat"].iloc[i]      + frac * (df["lat"].iloc[i + 1]      - df["lat"].iloc[i])
                fill["lon"]          = df["lon"].iloc[i]      + frac * (df["lon"].iloc[i + 1]      - df["lon"].iloc[i])
                fill["Altitude"]     = df["Altitude"].iloc[i] + frac * (df["Altitude"].iloc[i + 1] - df["Altitude"].iloc[i])
                fill["Speed"]        = df["Speed"].iloc[i]    + frac * (df["Speed"].iloc[i + 1]    - df["Speed"].iloc[i])
                fill["interpolated"] = True  # mark as estimated, not observed

                rows.append(fill)

    rows.append(df.iloc[-1].to_dict())  # always keep the last real row

    result = pd.DataFrame(rows).reset_index(drop=True)
    result["interpolated"] = result.get("interpolated", False).fillna(False)

    return result

In [5]:
# DATA LOADING

def load_aircraft_data(folder_path):
    """
    Load every .csv in the folder into a dict:  { callsign: DataFrame }

    Handles two file formats:
    1. One aircraft per file  — callsign taken from filename
    2. Combined file          — must have a 'Callsign' column; split by aircraft
    """
    aircraft_data = {}

    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    print(f"Found {len(csv_files)} CSV file(s) in {folder_path}")

    for filename in csv_files:
        path = os.path.join(folder_path, filename)
        df   = pd.read_csv(path)

        # Parse timestamps
        df["UTC"] = pd.to_datetime(df["UTC"])

        # FlightRadar24 stores position as "lat,lon" in one column — split it
        df[["lat", "lon"]] = df["Position"].str.split(",", expand=True).astype(float)

        # Add interpolated flag (all False until fill_gaps adds True rows)
        df["interpolated"] = False

        if "Callsign" in df.columns:
            # Combined file — split by callsign
            for callsign, group in df.groupby("Callsign"):
                cleaned = group.sort_values("UTC").reset_index(drop=True)
                cleaned = fill_gaps(cleaned, GAP_THRESHOLD_SECONDS)
                aircraft_data[callsign] = cleaned
                n_interp = cleaned["interpolated"].sum()
                print(f"  [{filename}] {callsign}: {len(cleaned)} rows ({n_interp} interpolated)")
        else:
            # Single-aircraft file — use filename as callsign
            callsign = filename.replace(".csv", "")
            cleaned  = df.sort_values("UTC").reset_index(drop=True)
            cleaned  = fill_gaps(cleaned, GAP_THRESHOLD_SECONDS)
            aircraft_data[callsign] = cleaned
            n_interp = cleaned["interpolated"].sum()
            print(f"  [{filename}] {callsign}: {len(cleaned)} rows ({n_interp} interpolated)")

    return aircraft_data

In [6]:
# TFR ZONE

class TFRZone:
    """
    Circular Temporary Flight Restriction zone.

    In real operations this is a cylinder of airspace created before
    a launch — aircraft must stay outside the radius.

    Future extension: make the radius dynamic (expanding/contracting
    during launch) to model different TFR phases.
    """

    def __init__(self, lat, lon, radius_nm):
        self.lat    = lat
        self.lon    = lon
        self.radius = radius_nm

    def contains(self, lat, lon):
        """True if the position is INSIDE the TFR (i.e. a violation)."""
        return haversine(lat, lon, self.lat, self.lon) < self.radius

    def distance_to_edge(self, lat, lon):
        """
        Signed distance to TFR boundary (nm).
        Negative = inside (violation), Positive = outside (safe).
        """
        return haversine(lat, lon, self.lat, self.lon) - self.radius

In [7]:
# AIRCRAFT AGENT

class AircraftAgent(Agent):
    """
    One aircraft in the simulation.

    MODE: 'replay'
        Follows CSV trajectory row by row — pure historical replay.
        Used as the baseline to compare against policy performance.

    MODE: 'policy'
        Navigates autonomously using a rule-based decision process:
        1. Look 20 steps ahead for TFR conflicts → turn away
        2. Look 20 steps ahead for separation conflicts → turn away
        3. Otherwise → fly toward destination

        This is the MDP "rule-based policy" baseline before we add
        the learned Q-value / POMDP solver.
    """

    def __init__(self, unique_id, model, trajectory, mode="replay"):
        super().__init__(unique_id, model)

        self.trajectory = trajectory
        self.step_index = 0
        self.mode       = mode

        # ── Physical state (initialised from first data row) ──
        row0           = trajectory.iloc[0]
        self.lat       = float(row0["lat"])
        self.lon       = float(row0["lon"])
        self.altitude  = float(row0["Altitude"])
        self.speed     = float(row0["Speed"])

        # Heading from first two rows (or 0 if only one row)
        if len(trajectory) > 1:
            row1          = trajectory.iloc[1]
            self.heading  = bearing_between(row0["lat"], row0["lon"],
                                            row1["lat"], row1["lon"])
        else:
            self.heading  = float(row0.get("Direction", 0))

        # ── Destination (used in policy mode) ──
        last_row              = trajectory.iloc[-1]
        self.destination_lat  = float(last_row["lat"])
        self.destination_lon  = float(last_row["lon"])

        # ── Nominal values (used to reset after avoidance) ──
        self.nominal_speed    = float(trajectory["Speed"].mean())
        self.nominal_altitude = float(trajectory["Altitude"].mean())

        # ── Avoidance state (policy mode) ──
        self.is_avoiding               = False
        self.avoidance_steps_remaining = 0
        self.avoidance_action          = None  # 'tfr' or 'separation'

        # ── Status ──
        self.finished = False  # True once aircraft reaches destination / runs out of data

        # ── History (used for plotting what actually happened) ──
        self.history_lat      = [self.lat]
        self.history_lon      = [self.lon]
        self.history_altitude = [self.altitude]
        self.history_time     = [0]

    # ─────────────────────────────────
    # CONFLICT PREDICTION
    # ─────────────────────────────────

    def predict_future_position(self, steps_ahead):
        """
        Where will THIS aircraft be in `steps_ahead` steps?

        Uses dead reckoning (constant heading + speed).
        Simple, but good enough for a 20-step lookahead horizon.

        In POMDP phase this becomes a BELIEF over future positions —
        a probability distribution rather than a single point.
        """
        lat = self.lat
        lon = self.lon

        for _ in range(steps_ahead):
            lat, lon = move_by_heading(lat, lon, self.heading,
                                       self.speed, STEP_DURATION_SECONDS)
        return lat, lon

    def tfr_conflict_ahead(self):
        """
        Scan the next CONFLICT_LOOKAHEAD_STEPS positions.
        Returns True if ANY future position is inside the TFR.
        """
        for step in range(1, CONFLICT_LOOKAHEAD_STEPS + 1):
            fut_lat, fut_lon = self.predict_future_position(step)
            if self.model.tfr.contains(fut_lat, fut_lon):
                return True
        return False

    def separation_conflicts_ahead(self):
        """
        Check whether any other aircraft will come within 5 nm (warning)
        within the lookahead horizon.

        Returns: list of conflicting AircraftAgent objects.
        (We use 5 nm warning threshold so the agent reacts before the
        3 nm hard limit is actually breached.)
        """
        threats = []

        for other in self.model.aircraft_agents:
            if other.unique_id == self.unique_id:
                continue
            if other.finished:
                continue

            for step in range(1, CONFLICT_LOOKAHEAD_STEPS + 1):
                my_lat,    my_lon    = self.predict_future_position(step)
                their_lat, their_lon = other.predict_future_position(step)

                if haversine(my_lat, my_lon, their_lat, their_lon) < 5:
                    threats.append(other)
                    break  # one conflict with this aircraft is enough

        return threats

    # ─────────────────────────────────
    # AVOIDANCE HELPER
    # ─────────────────────────────────

    def heading_away_from(self, threat_lat, threat_lon):
        """
        Simple avoidance manoeuvre: turn 90° to the RIGHT relative to the
        direct line toward the threat.

        Example: threat is to the North-East (bearing 045°)
                 → we turn to heading 045 + 90 = 135° (South-East)

        In the MDP phase this will be replaced by the policy's action
        selection from {climb, descend, turn_left, turn_right, slow}.
        """
        direct_bearing = bearing_between(self.lat, self.lon, threat_lat, threat_lon)
        return (direct_bearing + 90) % 360

    # ─────────────────────────────────
    # STEP IMPLEMENTATIONS
    # ─────────────────────────────────

    def step_replay(self):
        """
        Deterministic baseline: advance one row in the CSV data.
        No decisions, no avoidance — pure replay.
        """
        if self.step_index < len(self.trajectory) - 1:
            self.step_index += 1
            row = self.trajectory.iloc[self.step_index]

            # Update heading BEFORE moving so history is consistent
            self.heading  = bearing_between(self.lat, self.lon,
                                            float(row["lat"]), float(row["lon"]))

            self.lat      = float(row["lat"])
            self.lon      = float(row["lon"])
            self.altitude = float(row["Altitude"])
            self.speed    = float(row["Speed"])
        else:
            self.finished = True

    def step_policy(self):
        """
        Rule-based autonomous navigation.

        Decision tree (priority order):
        1. Already avoiding → keep current avoidance heading
        2. TFR conflict ahead → turn away from TFR
        3. Separation conflict ahead → turn away from closest threat
        4. Clear → fly direct to destination
        """
        if self.finished:
            return

        # Check if close enough to destination to stop
        if haversine(self.lat, self.lon, self.destination_lat, self.destination_lon) < 10:
            self.finished = True
            return

        # ── Decision ──

        if self.is_avoiding and self.avoidance_steps_remaining > 0:
            # Keep the avoidance heading already set, count down timer
            self.avoidance_steps_remaining -= 1

            if self.avoidance_steps_remaining == 0:
                # Avoidance window over — resume flying toward destination
                self.is_avoiding     = False
                self.avoidance_action = None

        elif self.tfr_conflict_ahead():
            # TFR conflict — highest priority
            self.is_avoiding               = True
            self.avoidance_steps_remaining = 15  # avoid for 15 × 30s = 7.5 minutes
            self.avoidance_action          = "tfr"
            self.heading                   = self.heading_away_from(self.model.tfr.lat,
                                                                     self.model.tfr.lon)

        else:
            threats = self.separation_conflicts_ahead()

            if threats:
                # Separation conflict — turn away from closest aircraft
                closest_threat = threats[0]
                self.is_avoiding               = True
                self.avoidance_steps_remaining = 10
                self.avoidance_action          = "separation"
                self.heading                   = self.heading_away_from(closest_threat.lat,
                                                                         closest_threat.lon)
            else:
                # All clear — fly toward destination
                self.is_avoiding  = False
                self.heading      = bearing_between(self.lat, self.lon,
                                                    self.destination_lat,
                                                    self.destination_lon)

        # ── Movement ──
        # Move based on current heading and speed
        self.lat, self.lon = move_by_heading(
            self.lat, self.lon,
            self.heading, self.speed,
            STEP_DURATION_SECONDS
        )

    def step(self):
        """Called by Mesa every simulation tick."""
        if self.mode == "replay":
            self.step_replay()
        else:
            self.step_policy()

        # Always record history so we can plot what actually happened
        self.history_lat.append(self.lat)
        self.history_lon.append(self.lon)
        self.history_altitude.append(self.altitude)
        self.history_time.append(self.model.time)

In [8]:
# AIR TRAFFIC MODEL

class AirTrafficModel(Model):
    """
    The simulation environment — manages all aircraft and global events.

    Why model.time is the global clock:
        Separation violations involve multiple aircraft simultaneously.
        If we used each agent's own step_index they'd be out of sync
        (different flights have different numbers of data rows).
        model.time increments once per environment step, so all
        agents share the same reference frame for safety events.
    """

    def __init__(self, aircraft_data, mode="replay"):
        super().__init__()
        self.schedule = BaseScheduler(self)
        self.time     = 0
        self.mode     = mode

        # TFR zone
        self.tfr = TFRZone(TFR_LAT, TFR_LON, TFR_RADIUS_NM)

        # ── Violation logs ──
        self.violations            = []  # list of (callsign, time, dist_nm)
        self.separation_violations = []  # list of (cs1, cs2, time, dist_nm)

        # ── "Currently in violation" sets prevent logging the same
        #    ongoing violation every single step.
        #    A violation is logged ONCE when it starts; cleared when it ends.
        self._active_tfr_violations = set()   # { callsign }
        self._active_sep_violations = set()   # { frozenset({cs1, cs2}) }

        # ── Create one agent per aircraft ──
        self.aircraft_agents = []
        for callsign, df in aircraft_data.items():
            agent = AircraftAgent(callsign, self, df, mode=mode)
            self.schedule.add(agent)
            self.aircraft_agents.append(agent)

        print(f"Model initialised: {len(self.aircraft_agents)} aircraft, mode='{mode}'")

    # ─────────────────────────────────
    # VIOLATION CHECKING
    # ─────────────────────────────────

    def _check_violations(self):
        """
        After all agents have moved, check for safety events.

        TFR violation  : aircraft inside the restricted zone
        Separation loss: two aircraft < 3 nm apart
        """

        # TFR check
        for ac in self.aircraft_agents:
            if ac.finished:
                continue

            d  = haversine(ac.lat, ac.lon, self.tfr.lat, self.tfr.lon)
            cs = ac.unique_id

            if d < self.tfr.radius:
                # In violation — only log if not already active
                if cs not in self._active_tfr_violations:
                    self.violations.append((cs, self.time, round(d, 2)))
                    self._active_tfr_violations.add(cs)
                    print(f"  [t={self.time}] ⚠  TFR violation: {cs} ({d:.1f} nm from centre)")
            else:
                # Exited violation zone
                self._active_tfr_violations.discard(cs)

        # Separation check (every unique PAIR)
        for i in range(len(self.aircraft_agents)):
            for j in range(i + 1, len(self.aircraft_agents)):
                a1 = self.aircraft_agents[i]
                a2 = self.aircraft_agents[j]

                if a1.finished or a2.finished:
                    continue

                d    = haversine(a1.lat, a1.lon, a2.lat, a2.lon)
                pair = frozenset([a1.unique_id, a2.unique_id])

                if d < 3:  # ICAO minimum radar separation: 3 nm
                    if pair not in self._active_sep_violations:
                        self.separation_violations.append(
                            (a1.unique_id, a2.unique_id, self.time, round(d, 2))
                        )
                        self._active_sep_violations.add(pair)
                        print(f"  [t={self.time}] ⚠  Separation: {a1.unique_id} — {a2.unique_id} ({d:.2f} nm)")
                else:
                    self._active_sep_violations.discard(pair)

    # ─────────────────────────────────
    # SIMULATION STEP
    # ─────────────────────────────────

    def step(self):
        """Advance by one simulation tick: move all aircraft, then check safety."""
        self.schedule.step()   # every agent's .step() is called
        self.time += 1
        self._check_violations()

    # ─────────────────────────────────
    # METRICS
    # ─────────────────────────────────

    def metrics(self):
        """Return a dict of key performance indicators for comparison."""
        return {
            "mode":                    self.mode,
            "aircraft_count":          len(self.aircraft_agents),
            "steps":                   self.time,
            "tfr_violations":          len(self.violations),
            "separation_violations":   len(self.separation_violations),
        }

In [9]:
# =============================================================================
# VISUALISATION
# =============================================================================

def plot_matplotlib(model):
    """
    Quick 2D matplotlib plot — useful for fast debugging.
    Uses aircraft.history_lat/lon so it shows what the simulation
    actually did (including avoidance manoeuvres in policy mode).
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor("#0d1b2a")

    # ── Flight paths ──
    ax1 = axes[0]
    ax1.set_facecolor("#0d1b2a")

    for ac in model.aircraft_agents:
        ax1.plot(ac.history_lon, ac.history_lat, linewidth=1.5, label=ac.unique_id)
        ax1.scatter(ac.history_lon[0],  ac.history_lat[0],  color="lime", zorder=5, s=50)
        ax1.scatter(ac.history_lon[-1], ac.history_lat[-1], color="red",  zorder=5, s=50)

    # TFR zone (rough degree conversion: 1 nm ≈ 1/60°)
    angles      = np.linspace(0, 2 * np.pi, 100)
    radius_deg  = model.tfr.radius / 60
    tfr_lons    = model.tfr.lon + radius_deg * np.cos(angles)
    tfr_lats    = model.tfr.lat + radius_deg * np.sin(angles)
    ax1.fill(tfr_lons, tfr_lats, color="red", alpha=0.15)
    ax1.plot(tfr_lons, tfr_lats, color="red", linewidth=1.2, linestyle="--", label="TFR")

    ax1.set_xlabel("Longitude", color="#c8eaf5")
    ax1.set_ylabel("Latitude",  color="#c8eaf5")
    ax1.set_title(f"Flight Paths — {model.mode} mode", color="#00c8e0")
    ax1.tick_params(colors="#c8eaf5")
    ax1.legend(facecolor="#0d1b2a", labelcolor="#c8eaf5", fontsize=7)
    ax1.grid(True, alpha=0.2)
    for sp in ax1.spines.values():
        sp.set_edgecolor("#1a3a5c")

    # ── Altitude profiles ──
    ax2 = axes[1]
    ax2.set_facecolor("#0d1b2a")

    for ac in model.aircraft_agents:
        ax2.plot(ac.history_time, ac.history_altitude, linewidth=1.5, label=ac.unique_id)

    ax2.set_xlabel("Simulation Step", color="#c8eaf5")
    ax2.set_ylabel("Altitude (ft)",   color="#c8eaf5")
    ax2.set_title("Altitude Profiles", color="#00c8e0")
    ax2.tick_params(colors="#c8eaf5")
    ax2.legend(facecolor="#0d1b2a", labelcolor="#c8eaf5", fontsize=7)
    ax2.grid(True, alpha=0.2)
    for sp in ax2.spines.values():
        sp.set_edgecolor("#1a3a5c")

    plt.tight_layout()
    plt.show()


def plot_plotly(model, output_filename=None):
    """
    Interactive Plotly globe/map.

    KEY FIX vs. original: we now draw from aircraft.history_lat/lon
    (the actual simulation path) NOT from the raw CSV.
    This means that in policy mode you'll see avoidance deviations.
    In replay mode it matches the CSV exactly.
    """
    COLOURS = ["#00c8e0", "#ff9d00", "#00ff9d", "#ff4488",
               "#bf55ff", "#ffcc00", "#00ffcc", "#ff6600"]

    def rgba(hex_col, alpha=1.0):
        h = hex_col.lstrip("#")
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        return f"rgba({r},{g},{b},{alpha})"

    fig = go.Figure()

    # Collect all positions for map centering
    all_lats = [lat for ac in model.aircraft_agents for lat in ac.history_lat]
    all_lons = [lon for ac in model.aircraft_agents for lon in ac.history_lon]
    mid_lat  = (max(all_lats) + min(all_lats)) / 2
    mid_lon  = (max(all_lons) + min(all_lons)) / 2

    # ── Draw each aircraft ──
    for idx, ac in enumerate(model.aircraft_agents):
        col = COLOURS[idx % len(COLOURS)]

        # Faint trail line
        fig.add_trace(go.Scattergeo(
            lat=ac.history_lat, lon=ac.history_lon,
            mode="lines",
            line=dict(width=1.5, color=rgba(col, 0.35)),
            hoverinfo="skip",
            showlegend=False,
        ))

        # Position dots
        fig.add_trace(go.Scattergeo(
            lat=ac.history_lat, lon=ac.history_lon,
            mode="markers",
            marker=dict(size=4, color=col, opacity=0.85),
            text=[f"<b>{ac.unique_id}</b><br>Step: {t}<br>"
                  f"Alt: {int(alt):,} ft"
                  for t, alt in zip(ac.history_time, ac.history_altitude)],
            hoverinfo="text",
            hoverlabel=dict(bgcolor="#050c1c", bordercolor=col,
                            font=dict(color="#c8eaf5", size=12, family="Courier New")),
            name=ac.unique_id,
        ))

        # Departure (circle)
        fig.add_trace(go.Scattergeo(
            lat=[ac.history_lat[0]], lon=[ac.history_lon[0]],
            mode="markers",
            marker=dict(size=11, color=col, symbol="circle",
                        line=dict(color="white", width=2)),
            hovertext=f"DEPARTURE: {ac.unique_id}",
            hoverinfo="text", showlegend=False,
        ))

        # Arrival / end (square)
        fig.add_trace(go.Scattergeo(
            lat=[ac.history_lat[-1]], lon=[ac.history_lon[-1]],
            mode="markers",
            marker=dict(size=11, color=col, symbol="square",
                        line=dict(color="white", width=2)),
            hovertext=f"END: {ac.unique_id}",
            hoverinfo="text", showlegend=False,
        ))

    # ── TFR zone ring ──
    angles     = np.linspace(0, 2 * np.pi, 72)
    r_deg      = model.tfr.radius / 60  # approximate
    tfr_lats   = model.tfr.lat + r_deg * np.sin(angles)
    tfr_lons   = model.tfr.lon + r_deg * np.cos(angles)

    fig.add_trace(go.Scattergeo(
        lat=np.append(tfr_lats, tfr_lats[0]),
        lon=np.append(tfr_lons, tfr_lons[0]),
        mode="lines",
        line=dict(color="rgba(255,60,60,0.9)", width=2, dash="dash"),
        name="TFR Zone",
    ))

    fig.add_trace(go.Scattergeo(
        lat=[model.tfr.lat], lon=[model.tfr.lon],
        mode="markers+text",
        marker=dict(size=10, color="red", symbol="x"),
        text=["TFR"], textposition="top right",
        textfont=dict(color="red", size=11, family="Courier New"),
        hovertext=f"TFR Centre | Radius: {model.tfr.radius} nm",
        hoverinfo="text", showlegend=False,
    ))

    # ── Separation violation markers ──
    # Only mark the FIRST occurrence of each pair to avoid clutter
    seen = set()
    for v in model.separation_violations:
        pair = frozenset([v[0], v[1]])
        if pair in seen:
            continue
        seen.add(pair)

        a1 = next((a for a in model.aircraft_agents if a.unique_id == v[0]), None)
        a2 = next((a for a in model.aircraft_agents if a.unique_id == v[1]), None)

        if a1 and a2:
            t        = min(v[2], len(a1.history_lat) - 1, len(a2.history_lat) - 1)
            mid_vlat = (a1.history_lat[t] + a2.history_lat[t]) / 2
            mid_vlon = (a1.history_lon[t] + a2.history_lon[t]) / 2

            fig.add_trace(go.Scattergeo(
                lat=[mid_vlat], lon=[mid_vlon],
                mode="markers",
                marker=dict(size=14, color="yellow", symbol="triangle-up",
                            line=dict(color="orange", width=1.5)),
                hovertext=f"⚠ SEP: {v[0]} — {v[1]}<br>t={v[2]}, {v[3]:.2f} nm",
                hoverinfo="text", showlegend=False,
            ))

    # ── Map styling ──
    m = model.metrics()
    fig.update_geos(
        projection_type="mercator",
        center=dict(lat=mid_lat, lon=mid_lon),
        showland=True,       landcolor="#0d1b2a",
        showocean=True,      oceancolor="#060d1a",
        showlakes=True,      lakecolor="#081422",
        showcountries=True,  countrycolor="rgba(60,100,160,0.5)",
        showcoastlines=True, coastlinecolor="rgba(0,140,200,0.7)",
        lonaxis=dict(range=[min(all_lons) - 5, max(all_lons) + 5]),
        lataxis=dict(range=[min(all_lats) - 3, max(all_lats) + 3]),
        bgcolor="#030810",
    )
    fig.update_layout(
        title=dict(
            text=(f"Simulation — {m['mode'].upper()} | "
                  f"TFR violations: {m['tfr_violations']} | "
                  f"Sep violations: {m['separation_violations']}"),
            font=dict(color="#00c8e0", family="Courier New", size=13),
        ),
        paper_bgcolor="#030810",
        legend=dict(font=dict(color="#c8eaf5", family="Courier New"),
                    bgcolor="rgba(4,12,26,0.8)",
                    bordercolor="rgba(0,180,220,0.3)", borderwidth=1),
        margin=dict(l=0, r=0, t=50, b=0),
        height=800,
    )

    if output_filename is None:
        output_filename = f"aircraft_map_{model.mode}.html"

    fig.write_html(output_filename, config={"scrollZoom": True, "displaylogo": False})
    webbrowser.open("file://" + os.path.abspath(output_filename))
    return fig


def print_summary(model):
    """Clean terminal summary of simulation results."""
    m = model.metrics()
    print()
    print("=" * 55)
    print(f"  SIMULATION COMPLETE — {m['mode'].upper()} MODE")
    print("=" * 55)
    print(f"  Aircraft                : {m['aircraft_count']}")
    print(f"  Steps run               : {m['steps']}")
    print(f"  TFR violations          : {m['tfr_violations']}")
    print(f"  Separation violations   : {m['separation_violations']}")
    print("=" * 55)

    if model.violations:
        print("\n  TFR Violations:")
        for cs, t, d in model.violations:
            print(f"    {cs} at step {t} — {d} nm from TFR centre")

    if model.separation_violations:
        print("\n  Separation Violations:")
        for cs1, cs2, t, d in model.separation_violations:
            print(f"    {cs1} — {cs2} at step {t} — {d} nm apart")



In [ ]:

# =============================================================================
# MAIN — RUN BOTH MODES AND COMPARE
# =============================================================================

if __name__ == "__main__":

    # ── 1. Load data ──
    print("\nLoading aircraft data...")
    aircraft_data = load_aircraft_data(DATA_FOLDER)
    print(f"\nTotal aircraft: {len(aircraft_data)}")

    # Quick stats
    for cs, df in aircraft_data.items():
        dist = haversine(df["lat"].iloc[0], df["lon"].iloc[0],
                         df["lat"].iloc[-1], df["lon"].iloc[-1])
        n_gap = df["interpolated"].sum()
        print(f"  {cs}: {len(df)} rows, {dist:.0f} nm, {n_gap} interpolated gap-fill rows")

    # ── 2. Determine how long to run ──
    # Run until the longest flight is done
    max_steps = max(len(df) for df in aircraft_data.values())
    print(f"\nMax steps needed: {max_steps}")

    # ── 3. REPLAY mode (historical baseline) ──
    print("\n--- REPLAY (historical baseline) ---")
    replay_model = AirTrafficModel(aircraft_data, mode="replay")

    for _ in range(max_steps - 1):
        replay_model.step()

    print_summary(replay_model)

    # ── 4. POLICY mode (rule-based avoidance) ──
    print("\n--- POLICY (rule-based avoidance) ---")
    policy_model = AirTrafficModel(aircraft_data, mode="policy")

    for _ in range(max_steps - 1):
        policy_model.step()

    print_summary(policy_model)

    # ── 5. Side-by-side comparison ──
    rm = replay_model.metrics()
    pm = policy_model.metrics()
    print("\n--- COMPARISON (replay vs policy) ---")
    print(f"  TFR violations   :  REPLAY={rm['tfr_violations']}  →  POLICY={pm['tfr_violations']}")
    print(f"  Sep violations   :  REPLAY={rm['separation_violations']}  →  POLICY={pm['separation_violations']}")

    # ── 6. Visualise ──
    print("\nGenerating plots...")
    plot_matplotlib(replay_model)
    plot_plotly(replay_model, "aircraft_map_replay.html")
    plot_plotly(policy_model, "aircraft_map_policy.html")


Loading aircraft data...
Found 11 CSV file(s) in C:\Users\Dev Sharma\OneDrive\Documents\MDP\Datasets\Validation data (FlightRader24)
    Gap of 162s detected — inserting 4 interpolated points
    Gap of 258s detected — inserting 7 interpolated points
    Gap of 386s detected — inserting 11 interpolated points
    Gap of 70s detected — inserting 1 interpolated points
    Gap of 532s detected — inserting 16 interpolated points
  [AA1028_3960d8c8 (1).csv] AAL1028: 948 rows (39 interpolated)
    Gap of 718s detected — inserting 22 interpolated points
    Gap of 207s detected — inserting 5 interpolated points
    Gap of 149s detected — inserting 3 interpolated points
    Gap of 81s detected — inserting 1 interpolated points
    Gap of 71s detected — inserting 1 interpolated points
    Gap of 66s detected — inserting 1 interpolated points
    Gap of 108s detected — inserting 2 interpolated points
    Gap of 103s detected — inserting 2 interpolated points
    Gap of 132s detected — inserting